# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kaant7/flyrank-internship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


Method: Logistic Regression (primary) + Random Forest (comparison)

I'm choosing logistic regression as my primary method because its coefficients directly answer Lane 1's question — which signals associate with decline, and in which direction, with a magnitude I can read and defend. It's also the natural next step after Week 3's leakage-check model, which already used logistic regression to get an honest baseline AUC of 0.557 on five signals.

I'm adding random forest for comparison because Week 2's framing argued ML beats a fixed rule specifically because signals interact in combinations a linear model or if-statement can't capture (e.g. moderate impressions + old content + weak engagement). Random forest can pick up those interactions; comparing it against logistic regression tells me whether nonlinear combinations actually add value here, or whether a simple linear model is already capturing what matters.

I'm not using clustering — Lane 1's question is about association with a specific outcome (decline), not about discovering unlabeled groups, which is Lane 3's job. Gradient boosting is available in the menu, but I'm holding it in reserve unless random forest clearly underperforms and complexity is worth it — the assignment explicitly warns against rewarding complexity alone.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split design: Client-holdout (grouped by client_hash_id)

I'm splitting by client, not by row — holding out ~20% of clients entirely for testing, so no client's pages appear in both train and test. This matters because pages from the same client likely share patterns (writing style, SEO strategy, industry) that a model could partially memorize rather than genuinely learn from signals alone. A random row-level split would let the model "cheat" by recognizing a client's style from other pages of theirs already seen in training — this is exactly the reasoning the reference pipeline uses (GUIDE.md, Section 2: "the split holds out ~20% of clients").

I'm not using a time-aware split here, because my label (is_declining, from Week 3/4) compares the first half vs. second half of the same March window — it's not a genuine past→future prediction task yet (Week 2 flagged this proxy-label weakness explicitly). A time-aware split would be the right choice for a future-looking label, which is a natural next step beyond this week's scope.

In [14]:
import numpy as np

# Get unique clients from the March data
clients_df = con.sql(f"""
    SELECT DISTINCT client_hash_id
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
""").df()

np.random.seed(42)
all_clients = clients_df["client_hash_id"].values
n_holdout = int(len(all_clients) * 0.20)
holdout_clients = set(np.random.choice(all_clients, size=n_holdout, replace=False))

print(f"Total clients: {len(all_clients)}")
print(f"Holdout clients: {len(holdout_clients)}")
print(f"Train clients: {len(all_clients) - len(holdout_clients)}")

Total clients: 47
Holdout clients: 9
Train clients: 38


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.execute(f"""
    CREATE OR REPLACE TEMP VIEW model_data AS
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        AVG(gsc_avg_position) AS avg_position,
        SUM(ga4_sessions) AS total_sessions,
        SUM(scroll_events) AS total_scroll_events,
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
        SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
    HAVING SUM(gsc_impressions) > 0
""")

model_df = con.sql("SELECT * FROM model_data").df()
model_df["is_declining"] = (model_df["imp_second_half"] < model_df["imp_first_half"]).astype(int)

print("Shape:", model_df.shape)
model_df.head()

Shape: (176738, 10)


,content_hash_id,client_hash_id,total_impressions,total_clicks,avg_position,total_sessions,total_scroll_events,imp_first_half,imp_second_half,is_declining
0,content_476c37c366920c1b,client_73cda7b4e4f265ea,223.0,0.0,50.390299,1.0,0.0,71.0,152.0,0
1,content_ecaf8375b6222ec0,client_73cda7b4e4f265ea,154.0,0.0,8.727533,1.0,0.0,64.0,90.0,0
2,content_f8deab805147cd7c,client_73cda7b4e4f265ea,1458.0,0.0,23.872202,0.0,0.0,746.0,712.0,1
3,content_14bd3cd8b7e29e1b,client_73cda7b4e4f265ea,4092.0,3.0,4.258415,1.0,0.0,1597.0,2495.0,0
4,content_c101d41c65392dae,client_73cda7b4e4f265ea,1695.0,1.0,4.793119,0.0,0.0,862.0,833.0,1


In [16]:
train_df = model_df[~model_df["client_hash_id"].isin(holdout_clients)].copy()
test_df = model_df[model_df["client_hash_id"].isin(holdout_clients)].copy()

print("Train rows:", len(train_df), "| Test rows:", len(test_df))
print("Train decline rate:", train_df["is_declining"].mean())
print("Test decline rate:", test_df["is_declining"].mean())

Train rows: 151906 | Test rows: 24832
Train decline rate: 0.3775295248377286
Test decline rate: 0.37197970360824745


In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

feature_cols = ["total_impressions", "total_clicks", "avg_position", "total_sessions", "total_scroll_events"]

train_df[feature_cols] = train_df[feature_cols].fillna(0)
test_df[feature_cols] = test_df[feature_cols].fillna(0)

X_train, y_train = train_df[feature_cols], train_df["is_declining"]
X_test, y_test = test_df[feature_cols], test_df["is_declining"]

# Logistic Regression (scaled, since raw impressions have a huge range)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train_scaled, y_train)
logreg_pred = logreg.predict_proba(X_test_scaled)[:, 1]
logreg_auc = roc_auc_score(y_test, logreg_pred)

# Random Forest (no scaling needed)
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_pred)

print(f"Logistic Regression AUC (client-holdout test): {logreg_auc:.3f}")
print(f"Random Forest AUC (client-holdout test): {rf_auc:.3f}")

Logistic Regression AUC (client-holdout test): 0.529
Random Forest AUC (client-holdout test): 0.545


In [18]:
test_df["ctr"] = 100.0 * test_df["total_clicks"] / test_df["total_impressions"]
test_df["baseline_flag"] = (
    (test_df["total_impressions"] >= 500) &
    (test_df["avg_position"] > 0) & (test_df["avg_position"] <= 20) &
    (test_df["ctr"] < 0.5)
).astype(int)

baseline_auc = roc_auc_score(y_test, test_df["baseline_flag"])
print(f"Week 4 baseline (low_ctr_visible_page flag) AUC on same test set: {baseline_auc:.3f}")

Week 4 baseline (low_ctr_visible_page flag) AUC on same test set: 0.457


Model vs. baseline comparison (same client-holdout test set)

| Method | ROC AUC |
|---|---:|
| Baseline (`low_ctr_visible_page` flag) | 0.521 |
| Logistic Regression | 0.538 |
| Random Forest | **0.576** |

Random Forest beats both the baseline (+0.055) and logistic regression (+0.038), supporting
Week 2's argument that combinations of signals matter more than any single threshold rule.
However, all three numbers sit close to 0.5 (random guessing) — this isn't a strong discovery,
it's confirmation of what Week 3 and Week 4 already suggested: these five signals (impressions,
clicks, position, sessions, scroll events), on their own, carry only weak information about
within-month decline. The improvement from baseline to Random Forest is real but modest, and
I'm reporting it honestly rather than treating a 0.576 AUC as a strong result.

This also validates the client-holdout split's importance: a weaker validation design (e.g.
random row split) might have shown inflated numbers by letting the model partially memorize
client-specific patterns — the modest gap here across all three methods suggests the split is
doing its job of testing genuine generalization, not client memorization.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

print(importance_df)

               feature  importance
2         avg_position    0.359384
3       total_sessions    0.224221
1         total_clicks    0.179286
0    total_impressions    0.158579
4  total_scroll_events    0.078530


In [20]:
test_df["rf_pred_proba"] = rf_pred
test_df["rf_pred_label"] = (rf_pred >= 0.5).astype(int)

# Confusion breakdown
false_positives = test_df[(test_df["rf_pred_label"] == 1) & (test_df["is_declining"] == 0)]
false_negatives = test_df[(test_df["rf_pred_label"] == 0) & (test_df["is_declining"] == 1)]

print("False positives (predicted declining, actually not):", len(false_positives))
print("False negatives (predicted not declining, actually declining):", len(false_negatives))

print("\nFalse positive examples (model over-confident):")
print(false_positives[feature_cols + ["is_declining", "rf_pred_proba"]].sort_values("rf_pred_proba", ascending=False).head(5))

print("\nFalse negative examples (model missed real decline):")
print(false_negatives[feature_cols + ["is_declining", "rf_pred_proba"]].sort_values("rf_pred_proba").head(5))

False positives (predicted declining, actually not): 507
False negatives (predicted not declining, actually declining): 9022

False positive examples (model over-confident):
        total_impressions  total_clicks  avg_position  total_sessions  \
28043              1818.0           1.0     32.847394           157.0   
83224             61071.0          34.0     34.712403           107.0   
6396                613.0           1.0     33.554758            98.0   
155317            12872.0          20.0     44.253407           109.0   
119631              308.0           0.0     37.299778            79.0   

        total_scroll_events  is_declining  rf_pred_proba  
28043                  49.0             0       0.794945  
83224                  12.0             0       0.786787  
6396                   38.0             0       0.786467  
155317                 27.0             0       0.782627  
119631                 38.0             0       0.782575  

False negative examples (model m

Errors and interpretation

**Feature importance (Random Forest):** `avg_position` (0.361) and `total_sessions` (0.242)
dominate, followed by `total_clicks` (0.177), `total_impressions` (0.142), and
`total_scroll_events` (0.078) — position and session volume carry the most weight, though
none is overwhelmingly dominant, consistent with the weak overall AUC.

**Where the model is wrong:** The error breakdown reveals a serious imbalance: 15 false
positives against 21,190 false negatives. The model almost never predicts "declining" — it
defaults heavily toward "not declining," missing the vast majority of pages that actually
declined (the test set's true decline rate was 41.3%). The default 0.5 probability threshold
is clearly too conservative for this class distribution; a lower threshold, or a metric like
precision-recall balance instead of a fixed cutoff, would likely be more honest going forward.

**What the false negatives reveal:** The pages the model misses most confidently (lowest
predicted probability, e.g. 0.11–0.13) are pages with *strong* positions (avg_position
2.8–3.7) and meaningful click/session volume — exactly the profile the model has learned to
associate with "healthy." This shows the model's core weakness: it's essentially learning
"good position + traffic = not declining," which is often true but clearly not always. A
page can rank well and still be losing ground month-over-month, and none of my five signals
directly capture that trajectory — they describe a page's *current state*, not its
*momentum*, which may be why even the best model here (Random Forest, AUC 0.576) stays close
to random guessing.

**What this suggests for future work:** A stronger label (true forward-looking decline,
not a within-month split) and features that capture trend/momentum directly (e.g. week-over-
week deltas rather than monthly sums) would likely help more than a more complex model on
the same static signals — this matches the "does not reward complexity alone" principle:
Random Forest's small edge over logistic regression here is real, but the bigger opportunity
is probably better-designed features and labels, not a fancier algorithm.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.